# Structure of the legacy securities lending data

Source table `crp_sftds_ecb_legacy.state_sl_079`, the trade states before June 2026, to be used from 2021-01-01 to 2026-05-31. Same ESMA report as the new table, stored the old way, with the collateral as extra rows per piece via `local_index` instead of arrays. Five checks before the cleaning query is mapped onto it. Single day checks use 2026-05-29, the last business day of May 2026, and rely on the table being partitioned by `business_date`.

In [ ]:
import pyodbc
import pandas as pd
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

Which column is the table partitioned by? Single day filters are only cheap if it is `business_date`.

In [ ]:
query = f"""

SHOW PARTITIONS crp_sftds_ecb_legacy.state_sl_079

"""
df = pd.read_sql_query(query, cnxn)
df.tail(3)

## 1. What is a row, and what identifies a report?

Every report should have a row with `local_index = 0`, exactly one, and `techrcrdid` should be the same on all rows of a report.

In [ ]:
query = f"""

SELECT local_index, COUNT(*) AS n
FROM crp_sftds_ecb_legacy.state_sl_079
WHERE business_date = '2026-05-29'
GROUP BY 1
ORDER BY 1
LIMIT 25

"""
df = pd.read_sql_query(query, cnxn)
df

In [ ]:
query = f"""

SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT techrcrdid) AS n_reports,
       SUM(CASE WHEN local_index = 0 THEN 1 ELSE 0 END) AS n_row0,
       COUNT(DISTINCT CASE WHEN local_index = 0 THEN techrcrdid END) AS n_reports_with_row0,
       COUNT(DISTINCT tec_surrogate_key) AS n_surrogate_keys
FROM crp_sftds_ecb_legacy.state_sl_079
WHERE business_date = '2026-05-29'

"""
df = pd.read_sql_query(query, cnxn)
df

## 2. Which codes do the categorical fields use?

`direction` replaces `counterparty_side`, `is_opn_term` replaces `term_type`, `uncollsd` is a string where the new table has a boolean.

In [ ]:
query = f"""

SELECT 'direction' AS col, CAST(direction AS STRING) AS value, COUNT(*) AS n
FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
UNION ALL
SELECT 'is_opn_term', CAST(is_opn_term AS STRING), COUNT(*)
FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
UNION ALL
SELECT 'uncollsd', CAST(uncollsd AS STRING), COUNT(*)
FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
UNION ALL
SELECT 'rbtrate_type', CAST(rbtrate_type AS STRING), COUNT(*)
FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
UNION ALL
SELECT 'ctrctmod_lvl', CAST(ctrctmod_lvl AS STRING), COUNT(*)
FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
UNION ALL
SELECT 'ctrctmod_actntp', CAST(ctrctmod_actntp AS STRING), COUNT(*)
FROM crp_sftds_ecb_legacy.state_sl_079 WHERE business_date = '2026-05-29' AND local_index = 0 GROUP BY 1, 2
ORDER BY col, n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

## 3. How are quantity and price stored?

The legacy table splits units and nominal, and monetary, percentage and yield prices, into separate columns. The notation type of the new table has to be derived from which column is filled, using the new table's codes.

In [ ]:
query = f"""

SELECT CASE WHEN lndata_assttp_scty_qty IS NOT NULL THEN 1 ELSE 0 END AS has_units,
       CASE WHEN lndata_assttp_scty_nmnl_amt IS NOT NULL THEN 1 ELSE 0 END AS has_nominal,
       CASE WHEN lndata_assttp_scty_unitpric_amt IS NOT NULL THEN 1 ELSE 0 END AS has_price_amount,
       CASE WHEN lndata_assttp_scty_unitpric_pctg IS NOT NULL THEN 1 ELSE 0 END AS has_price_pct,
       CASE WHEN lndata_assttp_scty_unitpric_yld IS NOT NULL THEN 1 ELSE 0 END AS has_price_yield,
       COUNT(*) AS n
FROM crp_sftds_ecb_legacy.state_sl_079
WHERE business_date = '2026-05-29' AND local_index = 0
GROUP BY 1, 2, 3, 4, 5
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

The codes the new table uses for the same distinction.

In [ ]:
query = f"""

SELECT loan_security_quantity_or_nominal_amount_notation_type AS quantity_notation,
       loan_security_price_notation_type AS price_notation,
       COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE reference_period = (SELECT MAX(reference_period) FROM crp_sftds_ecb.trade_states_securitieslending)
GROUP BY 1, 2
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

## 4. Are the collateral component counts per report?

`collcmpnttp_scty` and `collcmpnttp_csh` should equal the number of rows of a report that carry a security or a cash piece.

In [ ]:
query = f"""

SELECT r.collcmpnttp_scty, r.n_sec_rows, r.collcmpnttp_csh, r.n_cash_rows, COUNT(*) AS n_reports
FROM (
  SELECT techrcrdid,
         MAX(collcmpnttp_scty) AS collcmpnttp_scty,
         MAX(collcmpnttp_csh) AS collcmpnttp_csh,
         SUM(CASE WHEN assttp_scty_id IS NOT NULL THEN 1 ELSE 0 END) AS n_sec_rows,
         SUM(CASE WHEN assttp_csh_amt IS NOT NULL THEN 1 ELSE 0 END) AS n_cash_rows
  FROM crp_sftds_ecb_legacy.state_sl_079
  WHERE business_date = '2026-05-29'
  GROUP BY techrcrdid
) r
GROUP BY 1, 2, 3, 4
ORDER BY n_reports DESC
LIMIT 30

"""
df = pd.read_sql_query(query, cnxn)
df

## 5. Does the deduplication key behave as in the new table?

Same group check as in the data structure notebook, on the row 0 rows only so that collateral rows do not count as legs, and on one day per year, the last business day of May from 2021 to 2026, instead of the full five years.

In [ ]:
query = f"""

SELECT n_rows, n_best, n_dates, COUNT(*) AS n_groups
FROM (
  SELECT ruti, business_date,
         COUNT(*) AS n_rows,
         SUM(CASE WHEN best_value_leg = 1 THEN 1 ELSE 0 END) AS n_best,
         COUNT(DISTINCT evtdt) AS n_dates
  FROM crp_sftds_ecb_legacy.state_sl_079
  WHERE business_date IN ('2021-05-31', '2022-05-31', '2023-05-31', '2024-05-31', '2025-05-30', '2026-05-29')
    AND local_index = 0
  GROUP BY ruti, business_date
) g
GROUP BY n_rows, n_best, n_dates
ORDER BY n_groups DESC

"""
df = pd.read_sql_query(query, cnxn)
df.head(30)

The rows without `ruti` should again be the net exposure collateral updates without UTI.

In [ ]:
query = f"""

SELECT ctrctmod_actntp, CASE WHEN uti IS NULL THEN 1 ELSE 0 END AS uti_missing, COUNT(*) AS n
FROM crp_sftds_ecb_legacy.state_sl_079
WHERE business_date = '2026-05-29' AND local_index = 0
  AND (ruti IS NULL OR ruti = '')
GROUP BY 1, 2
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df